# SpliceBERT Head–Context Atlas — Stage 2 (repaired)

Run **Runtime → Run all** on a CUDA GPU. This notebook restores the frozen discovery bundle, verifies the repaired Stage 2 overlay, resumes target shards in Drive, and withholds mechanism labels until both independent audits pass.

The Stage 2 output is `/MyDrive/gfm_atlas_stage2_v1_results`. It is separate from the already-complete discovery `gfm_atlas_screen_v1_results/target_shards`.

In [ ]:
# 1. Deterministic setup and import preflight
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys, zipfile
import torch
assert torch.cuda.is_available(), 'CUDA GPU required: Runtime -> Change runtime type -> GPU'

base_zip = Path('/content/drive/MyDrive/gfm_atlas_screen_colab_v1.zip')
drive_root = Path('/content/drive/MyDrive/gfm_atlas_screen_v1_results')
overlay = drive_root / 'gfm_atlas_stage2_colab_overlay_v3.zip'
sidecar = drive_root / 'gfm_atlas_stage2_colab_overlay_v3.zip.sha256.json'
unpack = Path('/content/gfm_atlas_screen')
repo = unpack / 'repo'
stage2_out = Path('/content/drive/MyDrive/gfm_atlas_stage2_v1_results')
assert base_zip.is_file(), base_zip
assert overlay.is_file() and sidecar.is_file(), (overlay, sidecar)

def sha256(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

# A clean local extraction makes Runtime -> Run all deterministic. Drive outputs are untouched.
if unpack.exists():
    shutil.rmtree(unpack)
with zipfile.ZipFile(base_zip) as archive:
    archive.extractall('/content')
base_manifest = json.loads((repo / 'BUNDLE_MANIFEST.json').read_text())
base_bad = [rel for rel, rec in base_manifest['files'].items()
            if not (repo / rel).is_file()
            or (repo / rel).stat().st_size != rec['size_bytes']
            or sha256(repo / rel) != rec['sha256']]
assert not base_bad, f'base bundle verification failed: {base_bad[:5]}'

expected_overlay = json.loads(sidecar.read_text())['sha256']
assert sha256(overlay) == expected_overlay, (sha256(overlay), expected_overlay)
with zipfile.ZipFile(overlay) as archive:
    archive.extractall(repo)
overlay_manifest = json.loads((repo / 'STAGE2_OVERLAY_MANIFEST.json').read_text())
assert overlay_manifest['schema'] == 'head_context_atlas.stage2_colab_overlay/3'
overlay_bad = [rel for rel, rec in overlay_manifest['files'].items()
               if not (repo / rel).is_file()
               or (repo / rel).stat().st_size != rec['size_bytes']
               or sha256(repo / rel) != rec['sha256']]
assert not overlay_bad, f'overlay verification failed: {overlay_bad[:5]}'

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r',
                str(repo / 'studies/splicebert_head_context_atlas/colab_atlas_package/requirements-colab.txt')],
               check=True)
entrypoint = repo / 'scripts/splicebert_head_context_atlas/run_atlas_stage2_mechanisms.py'
subprocess.run([sys.executable, str(entrypoint), '--help'], cwd=repo, check=True,
               stdout=subprocess.DEVNULL)

local_health = repo / 'results/splicebert_head_context_atlas/atlas_stage2_mechanism_v1'
stage2_out.mkdir(parents=True, exist_ok=True)
if not (stage2_out / 'run_manifest.health.json').is_file():
    shutil.copytree(local_health, stage2_out, dirs_exist_ok=True)
assert (stage2_out / 'run_manifest.health.json').is_file()
assert len(list((stage2_out / 'health_shards').glob('packet_*.json'))) == 256
print('SETUP_PASS | GPU:', torch.cuda.get_device_name(0), '| overlay:', expected_overlay)

In [ ]:
# 2. Resume/complete the untouched Stage 2 target phase
from pathlib import Path
import os, subprocess, sys, time
repo = Path('/content/gfm_atlas_screen/repo')
stage2_out = Path('/content/drive/MyDrive/gfm_atlas_stage2_v1_results')
entrypoint = repo / 'scripts/splicebert_head_context_atlas/run_atlas_stage2_mechanisms.py'
assert entrypoint.is_file(), 'Run setup cell 1 after any runtime restart'
assert (stage2_out / 'run_manifest.health.json').is_file(), 'Stage 2 health seed is missing'
env = os.environ.copy()
env['PYTHONHASHSEED'] = '0'
env['TOKENIZERS_PARALLELISM'] = 'false'
before = len(list((stage2_out / 'target_shards').glob('packet_*.json')))
print(f'Resuming Stage 2 target at {before}/4019 packets')
command = [
    sys.executable, str(entrypoint),
    '--package', 'results/splicebert_head_context_atlas/atlas_stage2_package_v1.json',
    '--package-audit', 'results/splicebert_head_context_atlas/atlas_stage2_package_v1.audit.json',
    '--stage1-package', 'results/splicebert_head_context_atlas/atlas_mechanism_package_v1.json',
    '--output-dir', str(stage2_out), '--phase', 'target', '--health-n', '256',
    '--batch', '8', '--device', 'cuda',
]
log_path = stage2_out / 'stage2_target_runtime.log'
with log_path.open('a', buffering=1) as log:
    header = f'\n--- launch {time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())} | python={sys.version.split()[0]} ---\n'
    print(header, end='')
    log.write(header)
    process = subprocess.Popen(
        command, cwd=repo, env=env, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, bufsize=1,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='')
        log.write(line)
    returncode = process.wait()
if returncode:
    tail = ''.join(log_path.read_text(errors='replace').splitlines(keepends=True)[-120:])
    raise RuntimeError(f'Stage 2 child failed with exit {returncode}. Full log: {log_path}\n\n{tail}')
after = len(list((stage2_out / 'target_shards').glob('packet_*.json')))
assert after == 4019, f'incomplete Stage 2 target: {after}/4019 packets'
assert (stage2_out / 'run_manifest.target.json').is_file()
print('STAGE2_TARGET_COMPLETE | packets:', after)

In [ ]:
# 3. Audit, merge, analyze, and independently reconstruct (fail closed)
from pathlib import Path
import json, os, subprocess, sys
repo = Path('/content/gfm_atlas_screen/repo')
stage2_out = Path('/content/drive/MyDrive/gfm_atlas_stage2_v1_results')
assert repo.is_dir(), 'Run setup cell 1 after any runtime restart'
assert len(list((stage2_out / 'target_shards').glob('packet_*.json'))) == 4019
env = os.environ.copy()
env['PYTHONHASHSEED'] = '0'
env['TOKENIZERS_PARALLELISM'] = 'false'
merged = stage2_out / 'atlas_stage2_mechanism_v1_merged'
analysis = stage2_out / 'atlas_stage2_mechanism_analysis_v1'
package = 'results/splicebert_head_context_atlas/atlas_stage2_package_v1.json'
package_audit = 'results/splicebert_head_context_atlas/atlas_stage2_package_v1.audit.json'
steps = [
    (stage2_out / 'validation_audit.json', [sys.executable, 'scripts/splicebert_head_context_atlas/validate_atlas_stage2_mechanisms.py', '--run-dir', str(stage2_out), '--package', package, '--package-audit', package_audit]),
    (Path(str(merged) + '.manifest.json'), [sys.executable, 'scripts/splicebert_head_context_atlas/merge_atlas_stage2_mechanisms.py', '--run-dir', str(stage2_out), '--package', package, '--prefix', str(merged)]),
    (Path(str(analysis) + '.summary.json'), [sys.executable, 'scripts/splicebert_head_context_atlas/analyze_atlas_stage2_mechanisms.py', '--merged-prefix', str(merged), '--package', package, '--package-audit', package_audit, '--execution-audit', str(stage2_out / 'validation_audit.json'), '--run-dir', str(stage2_out), '--prefix', str(analysis)]),
    (Path(str(analysis) + '.audit.json'), [sys.executable, 'scripts/splicebert_head_context_atlas/validate_atlas_stage2_analysis.py', '--merged-prefix', str(merged), '--analysis-prefix', str(analysis), '--package', package, '--run-dir', str(stage2_out)]),
]
for sentinel, command in steps:
    if sentinel.is_file():
        print('Existing immutable step:', sentinel.name)
    else:
        subprocess.run(command, cwd=repo, env=env, check=True)
final_audit = json.loads(Path(str(analysis) + '.audit.json').read_text())
assert final_audit['status'] == 'PASS' and final_audit['n_failures'] == 0, final_audit
print('STAGE2_ANALYSIS_AUDIT_PASS | failures: 0')
print('Mechanism labels may now be read from:', str(analysis) + '.relations.json')